In [1]:
import os
import sys
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score, roc_auc_score, log_loss, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay, roc_curve
)

import lightgbm as lgb
from catboost import CatBoostClassifier
import xgboost as xgb

import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print("Libraries loaded successfully.")

Libraries loaded successfully.


## 1. Data Loading & Inspection

In [2]:
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')
sample_sub = pd.read_csv('sample_submission.csv')

print(f"Train Dataset Shape: {train_df.shape}")
print(f"Test Dataset Shape:  {test_df.shape}")
print(f"Sample Submission Shape: {sample_sub.shape}")

train_df.head()

Train Dataset Shape: (691369, 14)
Test Dataset Shape:  (296302, 13)
Sample Submission Shape: (296302, 2)


In [3]:
print("--- Missing Values in Train --- ")
print(train_df.isnull().sum())

print("\n--- Target Class Balance --- ")
print(train_df['addicted_label'].value_counts(normalize=True))

plt.figure(figsize=(6, 4))
sns.countplot(data=train_df, x='addicted_label', palette='viridis')
plt.title('Target Variable Distribution (Addicted Label)')
plt.xlabel('Addicted Label (0 = No, 1 = Yes)')
plt.ylabel('Count')
plt.show()

## 3. Advanced Feature Engineering
Domain-specific features are constructed to capture behavioral intensity, screen-to-sleep ratios, notifications per active hour, and missing value indicators.

In [4]:
def engineer_features(df):
    out = df.copy()
    out['missing_count'] = df.isnull().sum(axis=1)
    
    screen = out['daily_screen_time_hours']
    social = out['social_media_hours']
    gaming = out['gaming_hours']
    work = out['work_study_hours']
    sleep = out['sleep_hours']
    opens = out['app_opens_per_day']
    notifs = out['notifications_per_day']
    weekend = out['weekend_screen_time']
    age = out['age']
    
    out['total_screen_activity'] = screen + social + gaming
    out['weekend_extra_screen'] = weekend - screen
    out['screen_sleep_ratio'] = screen / (sleep + 0.1)
    out['social_screen_ratio'] = social / (screen + 0.1)
    out['gaming_screen_ratio'] = gaming / (screen + 0.1)
    out['work_screen_ratio'] = work / (screen + 0.1)
    out['notification_app_ratio'] = notifs / (opens + 1.0)
    out['notifications_per_screen_hour'] = notifs / (screen + 0.1)
    out['app_opens_per_screen_hour'] = opens / (screen + 0.1)
    out['screen_minus_sleep'] = screen - sleep
    out['screen_work_total'] = screen + work
    out['free_time_estimate'] = 24.0 - (sleep.fillna(7) + screen.fillna(4) + work.fillna(4))
    out['social_gaming_ratio'] = social / (gaming + 0.1)
    out['leisure_screen'] = social.fillna(0) + gaming.fillna(0)
    out['leisure_screen_ratio'] = out['leisure_screen'] / (screen + 0.1)
    out['age_screen_interaction'] = age * screen
    
    return out

df_combined = pd.concat([train_df.drop(columns=['addicted_label']), test_df], ignore_index=True)
df_features = engineer_features(df_combined)

cat_cols = ['gender', 'stress_level', 'academic_work_impact']
for col in cat_cols:
    df_features[col] = df_features[col].astype('category')

X = df_features.iloc[:len(train_df)].drop(columns=['id'])
y = train_df['addicted_label'].astype(int)
X_test = df_features.iloc[len(train_df):].drop(columns=['id'])

print(f"Engineered {X.shape[1]} features for modeling.")

Engineered 29 features for modeling.


In [5]:
if os.path.exists('model_metrics.npz'):
    metrics = np.load('model_metrics.npz')
    y_true = metrics['y_true']
    oof_ens = metrics['oof_ens']
    oof_lgb = metrics['oof_lgb']
    oof_cb = metrics['oof_cb']
    oof_xgb = metrics['oof_xgb']
    preds_ens = metrics['preds_ens']
    best_thresh = float(metrics['best_thresh'])
    best_acc = float(metrics['best_acc'])
    print("Loaded 5-Fold Cross Validation out-of-fold metrics.")
else:
    print("Metrics file not found. Ensure train_ensemble.py has executed.")

## 5. Model Evaluation & Accuracy Summary

In [6]:
print("==================================================")
print("             MODEL PERFORMANCE SUMMARY            ")
print("==================================================")
print(f"LightGBM OOF Accuracy: {accuracy_score(y_true, (oof_lgb >= 0.5).astype(int)):.5f} | ROC-AUC: {roc_auc_score(y_true, oof_lgb):.5f}")
print(f"CatBoost OOF Accuracy: {accuracy_score(y_true, (oof_cb >= 0.5).astype(int)):.5f} | ROC-AUC: {roc_auc_score(y_true, oof_cb):.5f}")
print(f"XGBoost  OOF Accuracy: {accuracy_score(y_true, (oof_xgb >= 0.5).astype(int)):.5f} | ROC-AUC: {roc_auc_score(y_true, oof_xgb):.5f}")
print("--------------------------------------------------")
print(f"Ensemble OOF ROC-AUC:  {roc_auc_score(y_true, oof_ens):.5f}")
print(f"Ensemble OOF Log Loss: {log_loss(y_true, oof_ens):.5f}")
print(f"Default Threshold (0.50) Accuracy: {accuracy_score(y_true, (oof_ens >= 0.5).astype(int)):.5f}")
print(f"OPTIMAL THRESHOLD ({best_thresh:.4f}) HIGHEST ACCURACY: {best_acc:.5f}")
print("==================================================")

## 6. Confusion Matrix & ROC Curve Plots

In [7]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. Confusion Matrix
y_pred_binary = (oof_ens >= best_thresh).astype(int)
cm = confusion_matrix(y_true, y_pred_binary)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Not Addicted (0)', 'Addicted (1)'])
disp.plot(cmap=plt.cm.Blues, ax=axes[0], values_format='d')
axes[0].set_title(f'Confusion Matrix (Optimal Thresh={best_thresh:.3f}, Acc={best_acc:.4f})')

# 2. ROC Curve
fpr, tpr, _ = roc_curve(y_true, oof_ens)
auc_val = roc_auc_score(y_true, oof_ens)
axes[1].plot(fpr, tpr, color='darkorange', lw=2, label=f'Ensemble ROC Curve (AUC = {auc_val:.4f})')
axes[1].plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('Receiver Operating Characteristic (ROC)')
axes[1].legend(loc="lower right")

plt.tight_layout()
plt.show()

## 7. Generate Submission Files for test.csv

In [8]:
# 1. Standard probability submission file (matching sample_submission.csv format)
sub_prob = pd.DataFrame({'id': test_df['id'], 'addicted_label': preds_ens})
sub_prob.to_csv('submission.csv', index=False)
print(f"Successfully wrote submission.csv ({len(sub_prob)} rows)")
print(sub_prob.head(5))

# 2. Binary prediction submission file (final_submission.csv)
sub_bin = pd.DataFrame({'id': test_df['id'], 'addicted_label': (preds_ens >= best_thresh).astype(int)})
sub_bin.to_csv('final_submission.csv', index=False)
print(f"\nSuccessfully wrote final_submission.csv with binary predictions.")
print(sub_bin.head(5))